# 02 - Preprocessing: membangun data siap latih

Menjalankan keputusan yang dikunci di `01_eda.ipynb`:

    muat -> buang missing -> resolusi konflik label -> dedup NFKC-exact
    -> stratified split 70/15/15 -> clean_text (SETELAH split)
    -> guard anti-kebocoran -> simpan

Keluaran: `data/processed/{train,val,test}.csv`, `data/processed/metadata.json`,
dan `data/interim/data_clean.csv`.

Urutan operasi menentukan isi split, sehingga mengubahnya membatalkan seluruh
angka eksperimen yang sudah dihasilkan.

In [ ]:
import pandas as pd

from src.config import LABEL_COLUMN, RAW_TEXT_COLUMN, settings
from src.services.preprocessing import DatasetBuilder

raw = pd.read_csv(settings.raw_csv, index_col=0)[[RAW_TEXT_COLUMN, LABEL_COLUMN]]
print(f"baris mentah: {len(raw):,}")

builder = DatasetBuilder(seed=settings.random_seed)

## 1. Jalankan pipeline

In [ ]:
splits = builder.build(raw)

print(builder.counts)
print(builder.label_conflict)
for name, frame in splits.items():
    positif = int((frame[LABEL_COLUMN] == 1).sum())
    print(f"{name:5s}: {len(frame):5,} baris | kelas judi {positif:4,} "
          f"({positif / len(frame) * 100:.2f}%)")

`leakage_removed` adalah baris val/test yang `text_clean`-nya identik dengan
baris train. Dedup NFKC di awal bekerja pada teks asli, sehingga dua komentar
yang hanya berbeda pada URL atau nominal masih lolos sebagai baris terpisah;
setelah placeholder diterapkan keduanya menjadi identik dan menjadi kebocoran
nyata. Prioritas pembuangan train > val > test, jadi himpunan latih tidak
pernah berkurang.

## 2. Contoh transformasi teks

In [ ]:
contoh = splits["train"]
mask = contoh["text_clean"].str.contains(r"\[URL\]|\[MENTION\]|\[NUM\]", regex=True)
for _, row in contoh[mask].head(6).iterrows():
    print(f"  L{row[LABEL_COLUMN]} | RAW  : {str(row[RAW_TEXT_COLUMN])[:95]}")
    print(f"       | CLEAN: {row['text_clean'][:95]}\n")

## 3. Verifikasi anti-kebocoran

In [ ]:
teks = {name: set(frame["text_clean"]) for name, frame in splits.items()}
kunci = {name: set(frame["nfkc_key"]) for name, frame in splits.items()}

for kiri, kanan in (("train", "val"), ("train", "test"), ("val", "test")):
    print(f"{kiri}-{kanan}: nfkc_key {len(kunci[kiri] & kunci[kanan])} | "
          f"text_clean {len(teks[kiri] & teks[kanan])}")

Keenam angka harus nol. Kebocoran train-test membuat retrieval RM-c menemukan
sampel uji di dalam indeksnya sendiri, yang akan melebih-lebihkan hasilnya.

## 4. Class weight

In [ ]:
weights = builder.compute_class_weights(splits["train"][LABEL_COLUMN])
print(weights)

Dihitung dari split train saja. Menghitungnya dari seluruh data akan
membocorkan distribusi val/test ke dalam loss.

## 5. Gate reproduktibilitas

In [ ]:
import hashlib
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    sementara = Path(tmp)
    builder.write(splits, output_dir=sementara,
                  interim_path=sementara / "data_clean.csv", source=settings.raw_csv)
    baru = {
        name: hashlib.sha256((sementara / f"{name}.csv").read_bytes()).hexdigest()
        for name in ("train", "val", "test")
    }

lama = {}
for name in ("train", "val", "test"):
    path = settings.split_path(name)
    lama[name] = hashlib.sha256(path.read_bytes()).hexdigest() if path.exists() else None

identik = True
for name in ("train", "val", "test"):
    if lama[name] is None:
        print(f"{name:5s}: belum ada split lama, akan dibuat baru")
    else:
        cocok = baru[name] == lama[name]
        identik &= cocok
        print(f"{name:5s}: {'IDENTIK' if cocok else 'BERBEDA'}  "
              f"{baru[name][:16]} vs {lama[name][:16]}")

if any(lama.values()) and not identik:
    raise RuntimeError(
        "Split baru berbeda dengan split yang sudah dipakai eksperimen. "
        "Jangan timpa: periksa dulu apa yang berubah, karena seluruh angka "
        "hasil mengasumsikan pembagian baris yang lama."
    )

Sel di atas menghitung split ke folder sementara dan membandingkan SHA-256-nya
dengan berkas yang sudah ada. Menimpa split secara diam-diam akan membuat
seluruh riwayat run menjadi yatim: angka lama merujuk pembagian baris yang tidak
lagi bisa direproduksi.

## 6. Simpan

In [ ]:
written = builder.write(splits, source=settings.raw_csv)
for name, path in written.items():
    print(f"{name:9s}: {path}  ({path.stat().st_size / 1024:.1f} KB)")

In [ ]:
import json

metadata = json.loads(settings.metadata_path.read_text(encoding="utf-8"))
print(json.dumps({k: metadata[k] for k in ("counts", "label_conflict", "class_weights")},
                 indent=2, ensure_ascii=False))

## Ringkasan

Input model adalah kolom `text_clean`, bukan `textOriginal`. Karena preprocessing
menambahkan tiga special token, setiap model wajib memanggil
`resize_token_embeddings(len(tokenizer))`; hal itu sudah ditangani oleh factory
di `src/models/heads.py`.

Lanjut ke `03a_rma_finetune.ipynb`.